# 03 Audio Signal EDA

Purpose:
- inspect the raw voice recordings before full feature extraction
- identify sample-rate, duration, silence, and quality issues
- visualize representative and outlier signals

Primary questions:
- Are the recordings consistent in format and duration?
- Are there silence-heavy or noisy files?
- What obvious outliers should we flag before modeling?


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Fixed raw data location under backend/data
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'backend' / 'data'
RAW_DIR = DATA_DIR / 'audio_lanzhou_2015'
print('Using raw data dir:', RAW_DIR)
RAW_DIR

In [ ]:
# Audio signal EDA: sample recordings and compute durations / silence
from pathlib import Path
import random
try:
    import librosa
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pd
except Exception as e:
    print('Missing audio/plotting packages:', e)
    librosa = None

# RAW_DIR expected from earlier cell: backend/data/audio_lanzhou_2015
if not RAW_DIR.exists():
    raise FileNotFoundError(f'Raw data dir not found: {RAW_DIR}')
# collect one sample file per subject (up to 200 files)
subjects = sorted([p for p in RAW_DIR.iterdir() if p.is_dir()])
sample_paths = []
for subj in subjects:
    wavs = list(subj.rglob('*.wav'))
    if wavs:
        sample_paths.append(wavs[0])
    if len(sample_paths) >= 200:
        break
print('Collected', len(sample_paths), 'sample files')
records = []
for p in sample_paths:
    if librosa is None:
        records.append({'path': str(p), 'sr': None, 'duration': None, 'voiced_ratio': None})
        continue
    try:
        y, sr = librosa.load(p, sr=None)
        duration = len(y) / sr if sr else None
        # simple voiced ratio via amplitude thresholding
        energy = np.abs(y)
        voiced_frames = np.sum(energy > (0.01 * np.max(energy)))
        voiced_ratio = float(voiced_frames) / len(y) if len(y)>0 else 0.0
        records.append({'path': str(p), 'sr': sr, 'duration': duration, 'voiced_ratio': voiced_ratio})
    except Exception as e:
        records.append({'path': str(p), 'sr': None, 'duration': None, 'voiced_ratio': None, 'error': str(e)})

if 'pd' in globals() and pd is not None:
    df = pd.DataFrame(records)
    display(df.head())
    print('Duration stats:')
    display(df['duration'].describe())
    print('Sample rates:')
    display(df['sr'].value_counts(dropna=True).head())
    if 'plt' in globals():
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        df['duration'].dropna().hist(bins=30, ax=ax[0])
        ax[0].set_title('Durations (s)')
        df['voiced_ratio'].dropna().hist(bins=30, ax=ax[1])
        ax[1].set_title('Voiced ratio (simple energy threshold)')
        plt.show()
else:
    print('pandas not available; printed only count of sample files')

# display one waveform and spectrogram for a representative file if librosa available
if librosa is not None and sample_paths:
    p = sample_paths[0]
    y, sr = librosa.load(p, sr=None)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12, 3))
    plt.plot(y[:min(len(y), sr*10)])
    plt.title(f'Waveform (first 10s) - {p.name}')
    plt.show()
    # spectrogram
    S = librosa.stft(y[:sr*10])
    import librosa.display
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(librosa.amplitude_to_db(np.abs(S), ref=np.max), sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram (first 10s)')
    plt.show()